In [ ]:
# ============================================================
#  AI-Powered Resume Screener  (Mini HR Tool)
#  Author  : Narjes
#  Stack   : LangChain · PyMuPDF · ChromaDB · phi3 (Ollama) · Gradio
#  Version : 1.0
# ============================================================

# ── Standard Library ────────────────────────────────────────
import os
import logging

# ── LangChain ───────────────────────────────────────────────
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate

# ── UI ──────────────────────────────────────────────────────
import gradio as gr


# ════════════════════════════════════════════════════════════
#  CONFIGURATION  ← only change these if needed
# ════════════════════════════════════════════════════════════

RESUMES_DIR = "./resumes"  # put your .txt / .pdf resumes here
CHROMA_DIR = "./chroma_resumes_db"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "phi3"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
TOP_K = 6  # higher K = more candidates considered

PROMPT_TEMPLATE = """
You are an expert HR assistant helping screen resumes.

Using ONLY the resume content provided below, answer the recruiter's question.
- List suitable candidates clearly with their name if mentioned.
- Highlight the relevant skills or experience that match the query.
- If no candidates match, say: "No suitable candidates found for this requirement."
- Be concise and professional.

Resume Content:
{context}

Recruiter Question:
{question}

Answer:
"""


# ════════════════════════════════════════════════════════════
#  LOGGING
# ════════════════════════════════════════════════════════════

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)


# ════════════════════════════════════════════════════════════
#  PIPELINE SETUP
# ════════════════════════════════════════════════════════════


def load_documents(folder: str):
    """Load all .txt and .pdf resumes from a folder."""
    docs = []
    for filename in os.listdir(folder):
        filepath = os.path.join(folder, filename)
        if filename.endswith(".pdf"):
            loader = PyMuPDFLoader(filepath)
            docs.extend(loader.load())
            log.info("Loaded PDF resume: %s", filename)
        elif filename.endswith(".txt"):
            loader = TextLoader(filepath, encoding="utf-8")
            docs.extend(loader.load())
            log.info("Loaded TXT resume: %s", filename)
    log.info("Total resume documents loaded: %d", len(docs))
    return docs


def split_documents(documents):
    """Split documents into overlapping chunks for retrieval."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    chunks = splitter.split_documents(documents)
    log.info("Created %d chunks", len(chunks))
    return chunks


def build_vectorstore(chunks, embeddings):
    """Load existing ChromaDB or build a new one."""
    if os.path.exists(CHROMA_DIR):
        log.info("Loading existing vector store from '%s'", CHROMA_DIR)
        return Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)
    log.info("Building new vector store at '%s'", CHROMA_DIR)
    return Chroma.from_documents(chunks, embeddings, persist_directory=CHROMA_DIR)


def setup_pipeline():
    """Full pipeline initialisation. Returns (retriever, llm, prompt)."""
    documents = load_documents(RESUMES_DIR)
    chunks = split_documents(documents)
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
    vectorstore = build_vectorstore(chunks, embeddings)
    llm = Ollama(model=LLM_MODEL)
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": TOP_K},
    )
    prompt = PromptTemplate(
        input_variables=["context", "question"],
        template=PROMPT_TEMPLATE,
    )
    log.info("Resume Screener pipeline ready.")
    return retriever, llm, prompt

def score_resume(resume, job_desc):
    score = 0

    if "python" in resume.lower():
        score += 30
    if "experience" in resume.lower():
        score += 30
    if "degree" in resume.lower():
        score += 20

    return score


def score_breakdown(resume):
    skills = 30 if "python" in resume.lower() else 10
    experience = 30 if "experience" in resume.lower() else 10
    education = 20 if "degree" in resume.lower() else 5

    return {
        "skills": skills,
        "experience": experience,
        "education": education
    }

def get_strengths(resume):
    strengths = []

    if "python" in resume.lower():
        strengths.append("Strong Python skills")
    if "project" in resume.lower():
        strengths.append("Hands-on project experience")
    if "team" in resume.lower():
        strengths.append("Team collaboration experience")

    return strengths


def get_concerns(resume):
    concerns = []

    if "experience" not in resume.lower():
        concerns.append("Limited experience mentioned")
    if "leadership" not in resume.lower():
        concerns.append("No leadership experience found")

    return concerns

# ════════════════════════════════════════════════════════════
#  INITIALISE
# ════════════════════════════════════════════════════════════

retriever, llm, prompt = setup_pipeline()


# ════════════════════════════════════════════════════════════
#  CORE SCREENING FUNCTION
# ════════════════════════════════════════════════════════════


def screen_resumes(question: str) -> str:
    """Retrieve relevant resume chunks and return an LLM-generated screening result."""
    if not question.strip():
        return "⚠️  Please enter a requirement or question."

    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    prompt_text = prompt.format(context=context, question=question)
    answer = llm.invoke(prompt_text)

    source_lines = ["\n\n─── Matched Resumes ────────────────────────"]
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "Unknown")
        excerpt = doc.page_content[:200].strip().replace("\n", " ")
        source_lines.append(f"\n[{i}] 📄 {os.path.basename(source)}: {excerpt}…")

    return str(answer) + "\n".join(source_lines)


# ════════════════════════════════════════════════════════════
#  GRADIO UI
# ════════════════════════════════════════════════════════════


def build_ui() -> gr.Interface:
    return gr.Interface(
        fn=screen_resumes,
        inputs=gr.Textbox(
            label="Recruiter Query",
            placeholder="e.g. Find candidates with TensorFlow + AWS experience",
            lines=2,
        ),
        outputs=gr.Textbox(
            label="Screening Result",
            lines=14,
        ),
        title="🧑‍💼 AI Resume Screener",
        description=(
            "Upload resumes to the `resumes/` folder, then ask who matches your requirements.\n"
            f"**Model:** {LLM_MODEL} · **Embeddings:** {EMBEDDING_MODEL} · **Vector DB:** ChromaDB\n"
            f"**Resumes folder:** `{RESUMES_DIR}` — supports .txt and .pdf files."
        ),
        flagging_mode="never",
    )


# ════════════════════════════════════════════════════════════
#  ENTRY POINT
# ════════════════════════════════════════════════════════════

if __name__ == "__main__":
    log.info("Launching Resume Screener → http://127.0.0.1:7860")
    ui = build_ui()
    ui.launch(share=False, inline=True)

2026-04-09 18:31:32,611 [INFO] Loaded TXT resume: resume 4.txt
2026-04-09 18:31:32,614 [INFO] Loaded TXT resume: resume1.txt
2026-04-09 18:31:32,616 [INFO] Loaded TXT resume: resume2.txt
2026-04-09 18:31:32,616 [INFO] Loaded TXT resume: resume3.txt
2026-04-09 18:31:32,616 [INFO] Loaded TXT resume: resume5.txt
2026-04-09 18:31:32,624 [INFO] Total resume documents loaded: 5
2026-04-09 18:31:32,627 [INFO] Created 10 chunks
C:\Users\pc\AppData\Local\Temp\ipykernel_10496\1236667599.py:112: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
2026-04-09 18:31:32,631 [INFO] Use pytorch device_name: cpu
2026-04-09 18:31:32,633 [I

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


2026-04-09 18:31:38,485 [INFO] HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
2026-04-09 18:31:38,873 [INFO] HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
